In [1]:
import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
import os

In [5]:
def compute_l4_polygons(reef_box):
    rx1, ry1, rx2, ry2 = reef_box
    reef_width  = rx2 - rx1
    reef_height = ry2 - ry1

    L4_HEIGHT_RATIO = 0.12  # adjust based on your model’s reef box
    l4_top    = ry1
    l4_bottom = ry1 + reef_height * L4_HEIGHT_RATIO

    slot_width = reef_width / 6

    polys = []
    for s in range(6):
        x1 = rx1 + s * slot_width
        x2 = rx1 + (s + 1) * slot_width
        poly = np.array([
            [x1, l4_top],
            [x2, l4_top],
            [x2, l4_bottom],
            [x1, l4_bottom]
        ])
        polys.append(poly)

    return polys

In [ ]:


# ---------------- CONFIG ----------------
VIDEO_FOLDER = '../yolov8_model/videos'
MODEL_PATH   = '../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt'

SLOT_CAPACITIES = [3, 1, 2, 2, 1, 3]
NUM_SLOTS = len(SLOT_CAPACITIES)
NUM_REEFS = 2  # adjust if needed

MIN_CONF_CORAL = 0.35
MIN_CONF_REEF  = 0.60

TRACK_PERSISTENCE_FRAMES = 5   # frames inside a slot to count as scored
MAX_TRACK_HISTORY = 15         # for velocity / stability if you want later

CROP_BOTTOM_RATIO = 2/5        # same as your original: crop bottom 2/5
CUTOFF_SECONDS = 49            # ignore last 49 seconds



# ---------------- HELPERS ----------------
def point_in_poly(x, y, poly):
    """Return True if (x, y) is inside polygon poly (np.array of shape (N, 2))."""
    return cv2.pointPolygonTest(poly, (float(x), float(y)), False) >= 0


def get_center(box):
    x1, y1, x2, y2 = box
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))


def is_coral_label(model, cls_idx):
    return model.names[int(cls_idx)] == 'coral'


def is_reef_label(model, cls_idx):
    return model.names[int(cls_idx)] == 'reef'


# ---------------- MAIN ----------------
def process_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Could not open {video_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    crop_h  = int(height * CROP_BOTTOM_RATIO)
    start_y = height - crop_h

    cutoff_frame = total_frames - int(CUTOFF_SECONDS * fps)

    # State: reef slot occupancy and scoring
    reef_grids = [[0] * NUM_SLOTS for _ in range(NUM_REEFS)]
    total_points = 0

    # Track state per coral track_id
    # track_state[track_id] = {
    #   "history": deque([(cx, cy), ...]),
    #   "slot": (reef_idx, slot_idx) or None,
    #   "frames_in_slot": int
    # }
    track_state = defaultdict(lambda: {
        "history": deque(maxlen=MAX_TRACK_HISTORY),
        "slot": None,
        "frames_in_slot": 0,
        "counted": False,  # whether this track has already scored
    })

    current_frame_idx = 0

    # Use YOLO tracking API
    # persist=True keeps track IDs across frames
    while True:
        ret, frame = cap.read()
        if not ret or current_frame_idx >= cutoff_frame:
            break

        cropped = frame[start_y:height, 0:width]

        results = model.track(
            cropped,
            conf=min(MIN_CONF_CORAL, MIN_CONF_REEF),
            persist=True,
            verbose=False
        )

        if len(results) == 0 or results[0].boxes is None:
            current_frame_idx += 1
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy()
        clss  = results[0].boxes.cls.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()
        ids   = results[0].boxes.id

        if ids is None:
            # tracking failed this frame; skip
            current_frame_idx += 1
            continue

        ids = ids.cpu().numpy().astype(int)

        # Optional: detect reefs if you still want them from YOLO
        # but for L4_SLOTS we assume you pre-calibrated polygons.
        # reefs = [boxes[i] for i, c in enumerate(clss)
        #          if is_reef_label(model, c) and confs[i] >= MIN_CONF_REEF]

        reef_boxes = [boxes[i] for i, c in enumerate(clss)
              if is_reef_label(model, c) and confs[i] >= MIN_CONF_REEF]

        reef_boxes.sort(key=lambda b: b[0])  # left reef first, right reef second

        L4_SLOTS = {}
        for r_idx, reef_box in enumerate(reef_boxes):
            polys = compute_l4_polygons(reef_box)
            for s_idx, poly in enumerate(polys):
                L4_SLOTS[(r_idx, s_idx)] = poly
        # --- Update coral tracks ---
        for box, cls_idx, conf, track_id in zip(boxes, clss, confs, ids):
            if not is_coral_label(model, cls_idx):
                continue
            if conf < MIN_CONF_CORAL:
                continue

            cx, cy = get_center(box)
            state = track_state[track_id]
            state["history"].append((cx, cy))

            # If this track already scored once, we don't score it again
            if state["counted"]:
                continue

            # Determine which slot (if any) this coral is in
            current_slot = None
            for (reef_idx, slot_idx), poly in L4_SLOTS.items():
                if point_in_poly(cx, cy, poly):
                    current_slot = (reef_idx, slot_idx)
                    break

            prev_slot = state["slot"]

            if current_slot is None:
                # Not in any L4 slot this frame
                state["slot"] = None
                state["frames_in_slot"] = 0
            else:
                # In some slot
                if prev_slot is None or prev_slot != current_slot:
                    # Just entered this slot
                    state["slot"] = current_slot
                    state["frames_in_slot"] = 1
                else:
                    # Stayed in same slot
                    state["frames_in_slot"] += 1

                # Check scoring condition
                reef_idx, slot_idx = current_slot
                if (state["frames_in_slot"] >= TRACK_PERSISTENCE_FRAMES and
                        reef_grids[reef_idx][slot_idx] < SLOT_CAPACITIES[slot_idx]):
                    reef_grids[reef_idx][slot_idx] += 1
                    total_points += 4
                    state["counted"] = True  # prevent double-counting this coral

        current_frame_idx += 1

    cap.release()

    print(f"Total points: {total_points}")
    for r in range(NUM_REEFS):
        print(f"Reef {r}: {reef_grids[r]}")


def main():
    model = YOLO(MODEL_PATH)

    video_files = [
        f for f in os.listdir(VIDEO_FOLDER)
        if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
    ]

    if not video_files:
        print("No video files found.")
        return

    for video_file in video_files:
        video_path = os.path.join(VIDEO_FOLDER, video_file)
        print(f"\nProcessing video: {video_file}")
        process_video(video_path, model)


if __name__ == "__main__":
    main()


Processing video: Qualification 72 - 2025 Iowa Regional.mp4
Total points: 44
Reef 0: [3, 1, 0, 0, 1, 3]
Reef 1: [3, 0, 0, 0, 0, 0]

Processing video: Qualification 79 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4
Total points: 48
Reef 0: [3, 1, 2, 2, 1, 3]
Reef 1: [0, 0, 0, 0, 0, 0]

Processing video: Qualification 26 - 2025 Iowa Regional.mp4
Total points: 24
Reef 0: [0, 1, 0, 0, 0, 0]
Reef 1: [3, 1, 0, 0, 1, 0]

Processing video: Qualification 15 - 2025 Iowa Regional.mp4
Total points: 80
Reef 0: [3, 1, 2, 2, 1, 3]
Reef 1: [1, 1, 2, 1, 0, 3]

Processing video: Qualification 43 - 2025 FIRST Championship - Hopper Division presented by PwC.mp4
Total points: 72
Reef 0: [3, 1, 1, 0, 0, 2]
Reef 1: [2, 1, 2, 2, 1, 3]

Processing video: Qualification 56 - 2025 Central Missouri Regional.mp4
Total points: 84
Reef 0: [3, 1, 2, 2, 1, 3]
Reef 1: [3, 1, 1, 1, 1, 2]

Processing video: Final 1 - 2025 Central Missouri Regional.mp4
Total points: 84
Reef 0: [3, 1, 2, 2, 1, 0]
Reef 1: 